# PA3 · Task 3 — Test-time compute: sampling, selection, refinement and adaptive inference

**Inputs:** the supplied `ATML-PA3-Data` folder (32 rollouts/question for Qwen2.5-3B-Instruct on a 1,000-question MATH subset and Qwen2.5-VL-3B-Instruct on MathVista testmini). Put it at `WORK/ATML-PA3-Data` (or set `DATA_ROOT`). The loader prints every table's schema and maps columns automatically; if a field is named unexpectedly, add it to `COLUMN_OVERRIDES` (names are listed in the supplied README.MD).

**Steps implemented:** (1) join + token-alignment checks, answer groups and correctness (math-verify for MATH, MathVista answer-type/choice/precision rules), dev/eval split (200/800), 10 seeded orders, FirstAns / oracle coverage / distinct groups, sampling-vs-selection failure decomposition; (2) mean log-prob, self-certainty, self-consistency, highest-score and RWC selection, vision-only and VisConf (prefix-recalibrated) for MathVista; (3) within-question signal analysis, length-only control, confidence–vision correlation, tokenwise-fusion ablation and illustrative trajectories; (4) live MATH comparison of single-long / 3-parallel / sequential-refinement at a 3,072-token allowance (the only GPU step); (5) agreement-based and quality-aware adaptive stopping with dev-selected thresholds; (6) compute-allocation figures, failure cases, bootstrap CIs.

**Approximate T4 time:** offline steps 1–3, 5–6 are CPU-bound ≈ 20–40 min total (answer grouping with math-verify dominates). Step 4 live generation (Qwen2.5-3B FP16, batch 1, 50 questions × 5 generations) ≈ 1.5–2.5 h. **Total ≈ 2–3 h.** Step 4 is resumable per question.

In [1]:
import os, sys, subprocess
if not os.environ.get("PA3_SKIP_INSTALL"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "math-verify[antlr4_13_2]==0.9.0", "transformers>=4.56,<6",
                    "accelerate>=0.33", "pyarrow", "pandas", "scipy", "matplotlib", "datasets>=2.19"], check=True)

In [2]:
import os, sys, json, math, time, re, gc, random
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/ATML_PA3/task3")
elif IN_KAGGLE:
    WORK = Path("/kaggle/working/task3")
else:
    WORK = Path(os.environ.get("PA3_WORK", "./pa3_work/task3")).resolve()

# ---- supplied data: unzip ATML-PA3-Data.zip (in WORK) to fast local disk once per session ----
DATA_ZIP = WORK / "ATML-PA3-Data.zip"
DATA_ROOT = Path("/content/ATML-PA3-Data") if IN_COLAB else WORK / "ATML-PA3-Data"
if not DATA_ROOT.exists() and DATA_ZIP.exists():
    import zipfile
    print("unzipping", DATA_ZIP, "→", DATA_ROOT)
    zipfile.ZipFile(DATA_ZIP).extractall(DATA_ROOT)
if DATA_ROOT.exists():
    for p in sorted(DATA_ROOT.rglob("*")):
        if p.suffix.lower() in (".parquet", ".md") and "__MACOSX" not in p.parts:
            print(" ", p.relative_to(DATA_ROOT), f"{p.stat().st_size/1e6:.1f} MB")
    _readme = next((p for p in DATA_ROOT.rglob("*") if p.name.lower().startswith("readme") and "__MACOSX" not in p.parts), None)
    if _readme:
        print("--- README lines mentioning revision / prompt / system ---")
        for line in open(_readme, errors="ignore"):
            if re.search(r"revision|commit|hash|prompt|system|instruction|boxed|think", line, re.I):
                print("   ", line.rstrip())

SEED = 6304
BUDGETS = [1, 2, 4, 8, 16, 32]
N_ORDERS, N_DEV, N_LIVE = 10, 200, 50
N_BOOT = 1000
TAUS = [0.5, 0.625, 0.75, 0.875, 1.0]
GAMMAS = [0.0, 0.1, 0.25, 0.5]
CHECK_AT = [4, 8, 16]
COLUMN_OVERRIDES = {"MathVista": {"att_image": "img_attn_total",
                                  "att_prompt": "prompt_text_attn_total",
                                  "att_generated": "generated_text_attn_total"}}     # e.g. {"MATH": {"extracted": "pred"}, "MathVista": {"att_image": "image_attention_sum"}}
LIVE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
LIVE_REVISION = "aa8e72537993ba99e69dfaafa59ed015b17504d1"
LIVE_SYSTEM = "You are a helpful assistant."
LIVE_SUFFIX = "Think step by step inside <think> </think> tags, then give your final answer inside \\boxed{}."
LIVE_JOIN = "\n\n"         # problem + LIVE_JOIN + LIVE_SUFFIX (match README.MD if it specifies otherwise)
REVIEW_PROMPT = ("Review the solution above. Identify and correct any errors; if it is correct, retain its answer. "
                 "Give your final answer inside \\boxed{}.")
GEN_KW = dict(do_sample=True, temperature=0.7, top_p=0.8, top_k=20, repetition_penalty=1.0)
CAP_SINGLE, CAP_PARALLEL, CAP_REVISION = 3072, 1024, 2048

# tonight (CPU runtime): no live generation. Later on a T4: set RUN_LIVE = True and Run all.
RUN_OFFLINE, RUN_SIGNALS, RUN_ADAPTIVE, RUN_LIVE, RUN_ANALYSIS = True, True, True, True, True
OUT, FIG_DIR, TAB_DIR = WORK / "results", WORK / "figures", WORK / "tables"
for d in [WORK, OUT, FIG_DIR, TAB_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("WORK =", WORK, "| data root exists:", DATA_ROOT.exists())

Mounted at /content/drive
unzipping /content/drive/MyDrive/ATML_PA3/task3/ATML-PA3-Data.zip → /content/ATML-PA3-Data
  ATML-PA3-Data/README.MD 0.0 MB
  ATML-PA3-Data/qwen25-math/generations.parquet 26.9 MB
  ATML-PA3-Data/qwen25-math/probability_metrics.parquet 155.2 MB
  ATML-PA3-Data/qwen25vl-mathvista/attention_metrics.parquet 138.5 MB
  ATML-PA3-Data/qwen25vl-mathvista/generations.parquet 17.1 MB
  ATML-PA3-Data/qwen25vl-mathvista/probability_metrics.parquet 81.1 MB
--- README lines mentioning revision / prompt / system ---
    All per-token lists align with `generated_token_ids`. Entry `k` describes the prediction of token `k`, conditioned on the prompt and the preceding `k` generated tokens. Selected stop tokens are excluded from every list.
    For MATH, extraction finds the last literal `\boxed{` and scans balanced braces, preserving nested LaTeX and stripping surrounding whitespace. An absent, empty, or unclosed last box gives null; there is no numeric fallback. MathVista pred

In [3]:
def jdump(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1, default=lambda o: o.item() if hasattr(o, "item") else (o.tolist() if hasattr(o, "tolist") else str(o)))
    os.replace(tmp, path)

def jload(path):
    with open(path) as f:
        return json.load(f)

def append_jsonl(path, rec):
    with open(path, "a") as f:
        f.write(json.dumps(rec) + "\n")

def read_jsonl(path):
    if not Path(path).exists():
        return []
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

def is_missing(x):
    if x is None:
        return True
    if isinstance(x, float) and math.isnan(x):
        return True
    return isinstance(x, str) and x.strip() in ("", "None", "none", "null", "NaN")

def last_boxed(text):
    """Content of the last \\boxed{...} (balanced braces); None if absent or unbalanced."""
    if text is None:
        return None
    i = max(text.rfind("\\boxed{"), text.rfind("\\fbox{"))
    if i < 0:
        m = list(re.finditer(r"\\boxed\s+([^\s$]+)", text))
        return m[-1].group(1) if m else None
    j = text.index("{", i)
    depth, k = 0, j
    while k < len(text):
        if text[k] == "{":
            depth += 1
        elif text[k] == "}":
            depth -= 1
            if depth == 0:
                return text[j + 1:k]
        k += 1
    return None

## 1 · Load, join and align the supplied rollouts

In [4]:
SCHEMA_CANDIDATES = {
    "text": ["text", "response", "generation", "generated_text", "output", "completion", "output_text"],
    "token_ids": ["token_ids", "output_token_ids", "generated_token_ids", "gen_token_ids", "completion_token_ids", "tokens"],
    "n_tokens": ["num_tokens", "n_tokens", "num_generated_tokens", "num_output_tokens", "n_output_tokens", "output_len",
                 "gen_len", "generated_length", "length", "token_count", "n_gen_tokens", "completion_tokens"],
    "extracted": ["extracted_answer", "pred_answer", "extracted", "prediction", "pred", "answer_extracted", "final_answer",
                  "parsed_answer"],
    "gold": ["gold_answer", "gold", "reference_answer", "ground_truth", "answer", "target", "label", "solution_answer"],
    "problem": ["problem", "question", "query", "input", "prompt_text", "prompt"],
    "logp": ["logp", "token_logp", "logprobs", "log_probs", "sampled_logp", "token_logprobs"],
    "kl": ["kl_u_p", "kl", "kl_uniform", "token_kl"],
    "question_type": ["question_type"], "answer_type": ["answer_type"], "precision": ["precision"],
    "choices": ["choices", "options"],
}

def resolve(df, field, ds, required=True):
    ov = COLUMN_OVERRIDES.get(ds, {})
    if field in ov:
        return ov[field]
    cols = {c.lower(): c for c in df.columns}
    for cand in SCHEMA_CANDIDATES.get(field, []):
        if cand in cols:
            return cols[cand]
    if required:
        raise KeyError(f"[{ds}] cannot find column for '{field}' in {list(df.columns)} — set COLUMN_OVERRIDES['{ds}']['{field}']")
    return None

def resolve_attention(att, ds):
    ov = COLUMN_OVERRIDES.get(ds, {})
    listcols = [c for c in att.columns if c not in ("dataset", "sample_id", "rollout_id")]
    def pick(key, words):
        if key in ov:
            return ov[key]
        hits = [c for c in listcols if any(w in c.lower() for w in words)]
        if len(hits) != 1:
            raise KeyError(f"[{ds}] ambiguous/missing attention column for {key}: {hits} of {listcols} — set COLUMN_OVERRIDES")
        return hits[0]
    gen_hits = [c for c in listcols if "generat" in c.lower() or c.lower().startswith("gen") or "_gen" in c.lower() or "output" in c.lower()]
    prm_hits = [c for c in listcols if "prompt" in c.lower() and c not in gen_hits]
    img = pick("att_image", ["image", "img", "vision", "visual"])
    prm = ov.get("att_prompt") or (prm_hits[0] if len(prm_hits) == 1 else pick("att_prompt", ["prompt", "text_in"]))
    gen = ov.get("att_generated") or (gen_hits[0] if len(gen_hits) == 1 else pick("att_generated", ["generat"]))
    return img, prm, gen

def find_model_dirs(root):
    dirs = sorted({p.parent for p in Path(root).rglob("generations.parquet")})
    assert dirs, f"no generations.parquet under {root}"
    vlm = [d for d in dirs if (d / "attention_metrics.parquet").exists()]
    llm = [d for d in dirs if d not in vlm]
    assert len(vlm) == 1 and len(llm) == 1, f"expected one LLM and one VLM folder, got {dirs}"
    return {"MATH": llm[0], "MathVista": vlm[0]}

def to_arr(x, dtype=np.float64):
    if x is None:
        return np.zeros(0, dtype)
    return np.asarray(x, dtype=dtype)

def load_mathvista_meta():
    from datasets import load_dataset
    ds = load_dataset("AI4Math/MathVista", split="testmini")
    keep = [c for c in ["pid", "question_type", "answer_type", "precision", "choices", "answer"] if c in ds.column_names]
    return {str(r["pid"]): r for r in ds.select_columns(keep)}

def load_dataset_tables(name, d):
    gen = pd.read_parquet(d / "generations.parquet")
    prob = pd.read_parquet(d / "probability_metrics.parquet")
    print(f"\n[{name}] {d}\n generations: {gen.shape} {list(gen.columns)}\n probability_metrics: {prob.shape} {list(prob.columns)}")
    keys = ["dataset", "sample_id", "rollout_id"]
    df = gen.merge(prob, on=keys, how="inner", suffixes=("", "_prob"))
    assert len(df) == len(gen) == len(prob), f"join lost rows: {len(gen)}, {len(prob)} → {len(df)}"
    if (d / "attention_metrics.parquet").exists():
        att = pd.read_parquet(d / "attention_metrics.parquet")
        print(f" attention_metrics: {att.shape} {list(att.columns)}")
        df = df.merge(att, on=keys, how="inner", suffixes=("", "_att"))
        assert len(df) == len(gen), "attention join lost rows"
        a_img, a_prm, a_gen = resolve_attention(att, name)
    else:
        a_img = a_prm = a_gen = None
    df["sample_id"] = df["sample_id"].astype(str)
    c = {f: resolve(df, f, name, required=f in ("extracted", "gold", "logp", "kl"))
         for f in ["text", "token_ids", "n_tokens", "extracted", "gold", "problem", "logp", "kl",
                   "question_type", "answer_type", "precision", "choices"]}
    print(" column map:", {k: v for k, v in c.items() if v}, "| attention:", (a_img, a_prm, a_gen))
    return df, c, (a_img, a_prm, a_gen)

## 2 · Answer equivalence, groups and correctness

In [5]:
import math_verify
from math_verify import parse as mv_parse_raw, verify as mv_verify

def mv_parse(s):
    s = str(s).strip()
    for cand in (s if (s.startswith("$") or "\\boxed" in s) else f"${s}$", s):
        try:
            p = mv_parse_raw(cand)
            if p:
                return p
        except Exception:
            pass
    return []

def mv_equal(a, b):
    try:
        return bool(mv_verify(a, b)) or bool(mv_verify(b, a))
    except Exception:
        return False

def group_math(extracted):
    """Answer groups for one question (no gold involved). Returns group ids (0 = NO_ANSWER) and representatives."""
    reps, rep_parsed, gid, seen = ["NO_ANSWER"], [None], [], {}
    for a in extracted:
        if is_missing(a):
            gid.append(0); continue
        key = str(a).strip()
        if key in seen:
            gid.append(seen[key]); continue
        p = mv_parse(key)
        g = None
        for j in range(1, len(reps)):
            if (p and rep_parsed[j] and mv_equal(rep_parsed[j], p)) or key == reps[j]:
                g = j; break
        if g is None:
            reps.append(key); rep_parsed.append(p); g = len(reps) - 1
        seen[key] = g; gid.append(g)
    return gid, reps, rep_parsed

def grade_math(reps, rep_parsed, gold):
    gp = mv_parse(gold)
    return [False] + [bool(gp and p and mv_equal(gp, p)) or (str(gold).strip() == r) for r, p in zip(reps[1:], rep_parsed[1:])]

def _lev(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def _as_list(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return []
    if isinstance(x, str):
        try:
            v = json.loads(x)
            return list(v) if isinstance(v, list) else [x]
        except Exception:
            return [x]
    return list(x)

def mathvista_normalize(extraction, choices, question_type, answer_type, precision):
    """MathVista's normalize_extracted_answer, but multiple-choice answers are represented as option letters."""
    if is_missing(extraction):
        return None
    if question_type == "multi_choice":
        ex = str(extraction).strip()
        letter = re.findall(r"\(([a-zA-Z])\)", ex)
        if letter:
            ex = letter[0].upper()
        m1 = re.fullmatch(r"\(?([A-Za-z])\)?[.:]?", ex)
        if m1:
            ex = m1.group(1)
        letters = [chr(ord("A") + i) for i in range(len(choices))]
        if ex.upper() in letters and len(ex) == 1:
            return ex.upper()
        if not choices:
            return None
        d = [_lev(ex, str(c)) for c in choices]          # most-similar option (official rule)
        return letters[int(np.argmin(d))]
    try:
        if answer_type == "integer":
            return str(int(float(str(extraction).replace(",", ""))))
        if answer_type == "float":
            p = 2 if is_missing(precision) else int(float(precision))
            return str(round(float(str(extraction).replace(",", "")), p))
        return str(extraction).strip()
    except Exception:
        return None

def mathvista_gold(gold, choices, question_type, answer_type, precision):
    if question_type == "multi_choice":
        cs = [str(c) for c in choices]
        if str(gold) in cs:
            return chr(ord("A") + cs.index(str(gold)))
        return mathvista_normalize(gold, choices, question_type, answer_type, precision)
    return mathvista_normalize(gold, choices, question_type, answer_type, precision)

## 3 · Per-question arrays, splits and seeded orders

In [6]:
def prepare(name):
    cache = OUT / f"prepared_{name}.pkl"
    if cache.exists():
        return pd.read_pickle(cache)
    d = find_model_dirs(DATA_ROOT)[name]
    df, c, (a_img, a_prm, a_gen) = load_dataset_tables(name, d)
    meta = None
    if name == "MathVista" and not (c["question_type"] and c["choices"] and c["answer_type"]):
        print(" MathVista metadata not in the supplied files → loading AI4Math/MathVista testmini")
        meta = load_mathvista_meta()
    df = df.sort_values(["sample_id", "rollout_id"])
    qids = sorted(df["sample_id"].unique())
    Q = []
    align = Counter(); t0 = time.time()
    for qi, (qid, g) in enumerate(df.groupby("sample_id", sort=True)):
        g = g.sort_values("rollout_id")
        R = []
        for _, r in g.iterrows():
            lp, kl = to_arr(r[c["logp"]]), to_arr(r[c["kl"]])
            T = len(lp)
            tid_len = len(r[c["token_ids"]]) if c["token_ids"] and r[c["token_ids"]] is not None else None
            ntok = int(r[c["n_tokens"]]) if c["n_tokens"] and not is_missing(r[c["n_tokens"]]) else (tid_len if tid_len is not None else T)
            ok = (len(kl) == T) and (tid_len is None or tid_len in (T, T + 1)) and (ntok in (T, T + 1))
            align["ok" if ok else "mismatch"] += 1
            rid_ = r["rollout_id"]
            rid_ = int(rid_) if str(rid_).lstrip("-").isdigit() else str(rid_)
            rec = dict(rollout_id=rid_, ntok=ntok, T=T, logp=lp.astype(np.float32), kl=kl.astype(np.float32),
                       extracted=None if is_missing(r[c["extracted"]]) else str(r[c["extracted"]]),
                       text=r[c["text"]] if c["text"] else None)
            if a_img:
                ai, ap, ag = to_arr(r[a_img]), to_arr(r[a_prm]), to_arr(r[a_gen])
                L = min(len(ai), len(ap), len(ag), T)
                if not (len(ai) == len(ap) == len(ag) == T):
                    align["attention_len_mismatch"] += 1
                den = ai[:L] + ap[:L] + ag[:L]
                rec["v"] = np.where(den > 0, ai[:L] / np.where(den > 0, den, 1), 0.0).astype(np.float32)
            R.append(rec)
        r0 = g.iloc[0]
        gold = r0[c["gold"]]
        q = dict(qid=qid, gold=None if is_missing(gold) else str(gold), problem=r0[c["problem"]] if c["problem"] else None, rollouts=R)
        if name == "MATH":
            gid, reps, parsed = group_math([x["extracted"] for x in R])
            q["group"], q["reps"] = np.array(gid), reps
            q["group_correct"] = np.array(grade_math(reps, parsed, q["gold"]))
        else:
            m = meta[qid] if meta is not None else r0
            qt = m["question_type"] if meta is not None else r0[c["question_type"]]
            at = m["answer_type"] if meta is not None else r0[c["answer_type"]]
            pr = m["precision"] if meta is not None else (r0[c["precision"]] if c["precision"] else None)
            pr = None if is_missing(pr) else pr
            ch = _as_list(m["choices"] if meta is not None else r0[c["choices"]])
            keys = [None if x["extracted"] is None else
                    (mathvista_normalize(x["extracted"], ch, qt, at, pr) or f"INVALID:{x['extracted'].strip()}") for x in R]
            reps = ["NO_ANSWER"]; gid = []
            for k in keys:
                if k is None:                       # only *missing* extractions form the NO_ANSWER group
                    gid.append(0)
                else:
                    if k not in reps:
                        reps.append(k)
                    gid.append(reps.index(k))
            gold_key = mathvista_gold(q["gold"], ch, qt, at, pr)
            q.update(group=np.array(gid), reps=reps, group_correct=np.array([False] + [r == gold_key for r in reps[1:]]),
                     question_type=qt, answer_type=at, precision=pr, choices=ch, gold_key=gold_key)
        q["correct"] = q["group_correct"][q["group"]]
        q["ntok"] = np.array([x["ntok"] for x in R])
        q["L"] = np.array([x["logp"].mean() if x["T"] else -np.inf for x in R])
        q["K"] = np.array([x["kl"].mean() if x["T"] else -np.inf for x in R])
        if name == "MathVista":
            q["V"] = np.array([x["v"].mean() if len(x["v"]) else 0.0 for x in R])
        Q.append(q)
        if qi % 200 == 0:
            print(f"  graded {qi}/{len(qids)} questions ({time.time()-t0:.0f}s)")
    n_roll = Counter(len(q["rollouts"]) for q in Q)
    print(f" alignment: {dict(align)} | rollouts per question: {dict(n_roll)}")
    # ---- split: lexicographic sort → default_rng(6304) permutation → first 200 dev, rest eval ----
    ids = np.array(qids)
    perm = np.random.default_rng(SEED).permutation(len(ids))
    ordered = ids[perm].tolist()
    split = {q: ("dev" if i < N_DEV else "eval") for i, q in enumerate(ordered)}
    for q in Q:
        q["split"] = split[q["qid"]]
    # ---- 10 seeded orders per question (sorted question order) ----
    rng = np.random.default_rng(SEED)
    for q in Q:
        n = len(q["rollouts"])
        q["orders"] = np.stack([rng.permutation(n) for _ in range(N_ORDERS)])
    info = dict(dataset=name, n_questions=len(Q), alignment=dict(align), rollouts_per_question=dict(n_roll),
                scorer=dict(math_verify=__import__("importlib.metadata").metadata.version("math-verify")) if name == "MATH" else
                dict(rules="MathVista normalize_extracted_answer (multi-choice → option letter, integer, float@precision, list)"),
                no_answer_rate=float(np.mean([np.mean(q["group"] == 0) for q in Q])),
                mean_tokens=float(np.mean([q["ntok"].mean() for q in Q])))
    jdump(dict(order=ordered, dev=ordered[:N_DEV], eval=ordered[N_DEV:], live=ordered[N_DEV:N_DEV + N_LIVE] if name == "MATH" else None,
               info=info), OUT / f"splits_{name}.json")
    jdump({q["qid"]: q["orders"].tolist() for q in Q}, OUT / f"orders_{name}.json")
    pd.to_pickle(Q, cache)
    print(" prepared:", info)
    return Q

In [7]:
DATASETS = ["MATH", "MathVista"]
if RUN_OFFLINE or RUN_SIGNALS or RUN_ADAPTIVE or RUN_LIVE:
    DATA = {name: prepare(name) for name in DATASETS}

In [8]:
# fill in MATH problem texts (the supplied column is "question_text", which the loader missed)
from datasets import load_dataset
_ds = load_dataset("DigitalLearningGmbH/MATH-lighteval", split="test",
                   revision="0530c78699ea5e8eb5530600900e1f328b48acad")
_live = set(jload(OUT / "splits_MATH.json")["live"])
_ok = _n = 0
for q in DATA["MATH"]:
    row = _ds[int(str(q["qid"]).split(":")[-1])]
    q["problem"] = row["problem"]
    if q["qid"] in _live:
        _n += 1
        _ok += (last_boxed(row["solution"]) or "").replace(" ", "") == (q["gold"] or "").replace(" ", "")
print(f"problem texts filled; gold check on the {_n} live questions: {_ok}/{_n} match")
assert _ok >= 0.9 * _n, "row mapping looks wrong — stop and check"

README.md:   0%|          | 0.00/8.41k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.99MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.86MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

problem texts filled; gold check on the 50 live questions: 50/50 match


## 4 · Selection methods over nested prefixes (all budgets, all 10 orders)
Ties: earliest supporting rollout in the current order; RWC uses average ranks. VisConf is recalibrated on every prefix (pooled token KL, 5th/95th linear percentiles of the revealed candidates only).

In [9]:
from scipy.stats import rankdata, spearmanr

def sc_select(groups):
    """Self-consistency: most votes; ties → group whose first supporter appears earliest."""
    cnt, first = Counter(), {}
    for i, g in enumerate(groups):
        cnt[g] += 1; first.setdefault(g, i)
    best = max(cnt.values())
    return min((first[g], g) for g, c in cnt.items() if c == best)[1], cnt

def rwc_select(groups, scores):
    N = len(groups)
    if N == 1:
        return groups[0], 1.0
    s = np.where(np.isfinite(scores), scores, -1e30)
    r = rankdata(-s, method="average")                 # rank 1 = best
    w = (N - r) / (N - 1)
    W, first = defaultdict(float), {}
    for i, (g, wi) in enumerate(zip(groups, w)):
        W[g] += wi; first.setdefault(g, i)
    vals = sorted(W.values(), reverse=True)
    best = vals[0]
    win = min((first[g], g) for g, v in W.items() if abs(v - best) <= 1e-12)[1]
    tot = sum(W.values())
    margin = (vals[0] - (vals[1] if len(vals) > 1 else 0.0)) / tot if tot > 0 else 0.0
    return win, margin

def visconf_scores(q, P):
    """Prefix-calibrated confidence C_i, VisConf Z_i = V_i C_i, tokenwise Z_i^token = mean_t c_it v_it."""
    R = q["rollouts"]
    pooled = np.concatenate([R[i]["kl"] for i in P]) if len(P) else np.zeros(0)
    if pooled.size == 0:
        z = np.zeros(len(P)); return z, z, z
    q05, q95 = np.percentile(pooled, [5, 95])          # numpy default = linear interpolation
    den = max(q95 - q05, 1e-8)
    C, Zt = np.zeros(len(P)), np.zeros(len(P))
    for k, i in enumerate(P):
        c = np.clip((R[i]["kl"] - q05) / den, 0, 1)
        C[k] = c.mean() if len(c) else 0.0
        v = R[i]["v"]; L = min(len(c), len(v))
        Zt[k] = (c[:L] * v[:L]).mean() if L else 0.0
    return C, q["V"][P] * C, Zt

def method_names(name):
    m = ["FirstAns", "SC", "MaxLogP", "MaxSelfCert", "SelfCert+RWC", "Shortest"]
    if name == "MathVista":
        m += ["Vision", "Vision+RWC", "VisConf", "VisConf+RWC", "VisConfTok", "VisConfTok+RWC"]
    return m

def evaluate_prefixes(name, Q):
    """Returns long DataFrame: one row per (question, order, N, method) + per-prefix stopping features."""
    cache = OUT / f"prefix_results_{name}.parquet"
    feat_cache = OUT / f"prefix_features_{name}.pkl"
    if cache.exists() and feat_cache.exists():
        return pd.read_parquet(cache), pd.read_pickle(feat_cache)
    rows, feats = [], {}
    t0 = time.time()
    for qi, q in enumerate(Q):
        gc_, grp, ntok = q["group_correct"], q["group"], q["ntok"]
        for o in range(N_ORDERS):
            order = q["orders"][o]
            for N in BUDGETS:
                P = order[:N]
                G = grp[P].tolist()
                base = dict(qid=q["qid"], split=q["split"], order=o, N=N, oracle=bool(q["correct"][P].any()),
                            n_groups=len(set(G)), cost=int(ntok[P].sum()))
                sel = {}
                sel["FirstAns"] = (P[0], int(ntok[P[0]]))
                g_sc, cnt = sc_select(G)
                sel["SC"] = (g_sc, None)
                for mname, sc in [("MaxLogP", q["L"]), ("MaxSelfCert", q["K"]), ("Shortest", -ntok.astype(float))]:
                    sel[mname] = (P[int(np.argmax(np.where(np.isfinite(sc[P]), sc[P], -1e30)))], None)
                g_rwc, marg_k = rwc_select(G, q["K"][P])
                sel["SelfCert+RWC"] = (g_rwc, None)
                if name == "MathVista":
                    C, Z, Zt = visconf_scores(q, P)
                    V = q["V"][P]
                    for mname, sc in [("Vision", V), ("VisConf", Z), ("VisConfTok", Zt)]:
                        sel[mname] = (P[int(np.argmax(sc))], None)
                        sel[mname + "+RWC"] = (rwc_select(G, sc)[0], None)
                    g_q, marg_q = rwc_select(G, Z)
                else:
                    g_q, marg_q = g_rwc, marg_k
                if N in CHECK_AT + [32]:
                    feats[(q["qid"], o, N)] = dict(sc=g_sc, sc_votes=cnt[g_sc], unique_lead=sum(1 for v in cnt.values() if v == cnt[g_sc]) == 1,
                                                   q_winner=g_q, q_margin=marg_q, cost=int(ntok[P].sum()))
                for mname in method_names(name):
                    v, c = sel[mname]
                    if N == 1:
                        v, c = P[0], int(ntok[P[0]])
                    if mname in ("SC", "SelfCert+RWC") or mname.endswith("+RWC"):
                        g = v if N > 1 else grp[P[0]]
                        rid = int(next(i for i in P if grp[i] == g))     # earliest supporting rollout
                    else:
                        rid = int(v); g = grp[rid]
                    rows.append(dict(base, method=mname, rollout=str(q["rollouts"][rid]["rollout_id"]), group=int(g),
                                     answer=q["reps"][g], correct=bool(gc_[g]),
                                     cost=c if mname == "FirstAns" else base["cost"]))
        if qi % 200 == 0:
            print(f"  [{name}] {qi}/{len(Q)} questions ({time.time()-t0:.0f}s)")
    df = pd.DataFrame(rows)
    df.to_parquet(cache, index=False); pd.to_pickle(feats, feat_cache)
    return df, feats

def summarize(df, split="eval"):
    d = df[df.split == split]
    per_q = d.groupby(["method", "N", "qid"]).agg(acc=("correct", "mean"), oracle=("oracle", "mean"),
                                                   cost=("cost", "mean"), n_groups=("n_groups", "mean")).reset_index()
    return per_q, per_q.groupby(["method", "N"]).agg(accuracy=("acc", "mean"), oracle=("oracle", "mean"),
                                                     mean_cost=("cost", "mean"), distinct_groups=("n_groups", "mean")).reset_index()

def paired_boot(a, b, n=N_BOOT, seed=SEED):
    a, b = np.asarray(a, float), np.asarray(b, float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(a), size=(n, len(a)))
    d = (a[idx] - b[idx]).mean(1)
    return float((a - b).mean()), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))

def failure_decomposition(df, split="eval"):
    d = df[df.split == split].copy()
    d["sampling_fail"] = ~d["oracle"]
    d["selection_fail"] = d["oracle"] & ~d["correct"]
    return d.groupby(["method", "N"]).agg(correct=("correct", "mean"), no_correct_candidate=("sampling_fail", "mean"),
                                          correct_available_but_missed=("selection_fail", "mean")).reset_index()

def run_offline():
    out = {}
    for name in DATASETS:
        df, feats = evaluate_prefixes(name, DATA[name])
        for split in ("dev", "eval"):
            per_q, summ = summarize(df, split)
            summ.to_csv(TAB_DIR / f"selection_{name}_{split}.csv", index=False)
            per_q.to_csv(TAB_DIR / f"per_question_{name}_{split}.csv", index=False)
        failure_decomposition(df).to_csv(TAB_DIR / f"failure_decomposition_{name}_eval.csv", index=False)
        per_q, summ = summarize(df, "eval")
        comp = []
        piv = per_q.pivot_table(index="qid", columns=["method", "N"], values="acc")
        for N in BUDGETS:
            for m in method_names(name):
                if m != "SC":
                    comp.append(dict(N=N, comparison=f"{m} − SC", **dict(zip(["diff", "ci_lo", "ci_hi"], paired_boot(piv[(m, N)], piv[("SC", N)])))))
            comp.append(dict(N=N, comparison="SC − FirstAns", **dict(zip(["diff", "ci_lo", "ci_hi"], paired_boot(piv[("SC", N)], piv[("FirstAns", N)])))))
            if name == "MathVista":
                for a, b in [("VisConf+RWC", "SelfCert+RWC"), ("VisConf+RWC", "VisConf"), ("SelfCert+RWC", "MaxSelfCert"),
                             ("VisConfTok+RWC", "VisConf+RWC")]:
                    comp.append(dict(N=N, comparison=f"{a} − {b}", **dict(zip(["diff", "ci_lo", "ci_hi"], paired_boot(piv[(a, N)], piv[(b, N)])))))
            else:
                comp.append(dict(N=N, comparison="SelfCert+RWC − MaxSelfCert", **dict(zip(["diff", "ci_lo", "ci_hi"],
                                 paired_boot(piv[("SelfCert+RWC", N)], piv[("MaxSelfCert", N)])))))
        pd.DataFrame(comp).to_csv(TAB_DIR / f"bootstrap_{name}_eval.csv", index=False)
        out[name] = (df, feats)
        show = summ[summ.N.isin([1, 4, 32])].pivot(index="method", columns="N", values="accuracy").round(4)
        print(f"\n[{name}] held-out accuracy (oracle@32 = {summ[(summ.method=='SC')&(summ.N==32)]['oracle'].iloc[0]:.4f})\n", show)
    return out

In [10]:
if RUN_OFFLINE or RUN_ADAPTIVE or RUN_ANALYSIS:
    OFF = run_offline()


[MATH] held-out accuracy (oracle@32 = 0.9137)
 N                 1       4       32
method                              
FirstAns      0.6435  0.6435  0.6435
MaxLogP       0.6435  0.6725  0.6400
MaxSelfCert   0.6435  0.6878  0.7062
SC            0.6435  0.6920  0.7379
SelfCert+RWC  0.6435  0.6970  0.7555
Shortest      0.6435  0.6521  0.6238

[MathVista] held-out accuracy (oracle@32 = 0.8950)
 N                   1       4       32
method                                
FirstAns        0.5364  0.5364  0.5364
MaxLogP         0.5364  0.5331  0.4825
MaxSelfCert     0.5364  0.5506  0.5475
SC              0.5364  0.5800  0.6312
SelfCert+RWC    0.5364  0.5699  0.6380
Shortest        0.5364  0.5518  0.5398
VisConf         0.5364  0.5630  0.5775
VisConf+RWC     0.5364  0.5796  0.6450
VisConfTok      0.5364  0.5588  0.5625
VisConfTok+RWC  0.5364  0.5780  0.6388
Vision          0.5364  0.5452  0.5375
Vision+RWC      0.5364  0.5676  0.6388


## 5 · What do the signals capture? (N = 32, within-question analysis)

In [11]:
def signal_analysis(name, Q):
    rows, corr_rows, cov_err = [], [], []
    for q in Q:
        if q["split"] != "eval":
            continue
        P = np.arange(len(q["rollouts"]))
        y = q["correct"].astype(float); ln = q["ntok"].astype(float)
        scores = {"mean_logp": q["L"], "self_certainty": q["K"], "shortest (−length)": -ln}
        if name == "MathVista":
            C, Z, Zt = visconf_scores(q, P)
            scores.update({"vision": q["V"], "calibrated_conf": C, "visconf": Z, "visconf_tokenwise": Zt})
            for i in P:   # within-rollout confidence–vision correlation
                R = q["rollouts"][i]
                q05, q95 = np.percentile(np.concatenate([r["kl"] for r in q["rollouts"]]), [5, 95])
                c = np.clip((R["kl"] - q05) / max(q95 - q05, 1e-8), 0, 1)
                L = min(len(c), len(R["v"]))
                if L > 2 and c[:L].std() > 0 and R["v"][:L].std() > 0:
                    corr_rows.append(dict(qid=q["qid"], rollout=i, corr=float(np.corrcoef(c[:L], R["v"][:L])[0, 1]),
                                          correct=bool(y[i])))
                if L:
                    cov = float(np.mean(c[:L] * R["v"][:L]) - np.mean(c[:L]) * np.mean(R["v"][:L]))
                    cov_err.append(abs((Zt[i] - Z[i]) - cov) if L == len(c) == len(R["v"]) else 0.0)
        mixed = 0 < y.mean() < 1
        for sname, s in scores.items():
            s = np.where(np.isfinite(s), s, -1e30)
            order = np.argsort(-s, kind="stable")
            k = len(s) // 4
            rec = dict(qid=q["qid"], score=sname, mixed=mixed, top_minus_bottom=y[order[:k]].mean() - y[order[-k:]].mean())
            if np.std(s) > 0 and np.std(y) > 0:
                rec["spearman_score_correct"] = spearmanr(s, y).statistic
            if np.std(s) > 0 and np.std(ln) > 0:
                rec["spearman_score_length"] = spearmanr(s, ln).statistic
            if np.std(ln) > 0 and np.std(y) > 0:
                rec["spearman_length_correct"] = spearmanr(ln, y).statistic
            rows.append(rec)
    df = pd.DataFrame(rows)
    summ = df.groupby("score").agg(top_minus_bottom_all=("top_minus_bottom", "mean"),
                                   spearman_score_correct=("spearman_score_correct", "mean"),
                                   spearman_score_length=("spearman_score_length", "mean"),
                                   spearman_length_correct=("spearman_length_correct", "mean")).reset_index()
    mixed = df[df.mixed].groupby("score")["top_minus_bottom"].mean().rename("top_minus_bottom_mixed_only")
    summ = summ.merge(mixed, on="score")
    summ["n_mixed_questions"] = int(df[df.mixed].qid.nunique())
    summ.to_csv(TAB_DIR / f"signals_{name}.csv", index=False)
    print(f"\n[{name}] signal analysis (held-out, N=32):\n", summ.round(4).to_string(index=False))
    if corr_rows:
        cr = pd.DataFrame(corr_rows); cr.to_csv(TAB_DIR / f"conf_vision_corr_{name}.csv", index=False)
        s = dict(mean=cr["corr"].mean(), median=cr["corr"].median(), frac_positive=(cr["corr"] > 0).mean(),
                 mean_correct=cr[cr.correct]["corr"].mean(), mean_incorrect=cr[~cr.correct]["corr"].mean(),
                 max_abs_identity_error_Ztok_minus_Z_vs_cov=float(max(cov_err) if cov_err else 0.0))
        jdump(s, TAB_DIR / f"conf_vision_corr_summary_{name}.json"); print(" confidence–vision correlation:", s)
        import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(5, 3.2))
        ax.hist(cr[cr.correct]["corr"], bins=40, alpha=0.6, label="correct rollouts", density=True)
        ax.hist(cr[~cr.correct]["corr"], bins=40, alpha=0.6, label="incorrect rollouts", density=True)
        ax.set_xlabel("within-rollout corr(c_t, v_t)"); ax.legend(fontsize=8); fig.tight_layout()
        fig.savefig(FIG_DIR / "conf_vision_corr_hist.png", dpi=200); plt.close(fig)
        illustrative_trajectories(Q)
    return summ

def illustrative_trajectories(Q, n=3):
    """Rule (fixed in advance): the first n dev questions in split-permutation order with both correct and incorrect
    rollouts; plot calibrated confidence c_t and visual engagement v_t for the highest-VisConf correct and incorrect rollout."""
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    order = jload(OUT / "splits_MathVista.json")["dev"]
    byid = {q["qid"]: q for q in Q}
    pick = [byid[x] for x in order if 0 < byid[x]["correct"].mean() < 1][:n]
    fig, axes = plt.subplots(len(pick), 2, figsize=(12, 2.6 * len(pick)), squeeze=False)
    sm = lambda x: np.convolve(x, np.ones(15) / 15, mode="valid") if len(x) > 15 else x
    for r, q in enumerate(pick):
        P = np.arange(len(q["rollouts"])); C, Z, Zt = visconf_scores(q, P)
        q05, q95 = np.percentile(np.concatenate([x["kl"] for x in q["rollouts"]]), [5, 95])
        for col, want in enumerate([True, False]):
            idx = [i for i in P if q["correct"][i] == want]
            i = max(idx, key=lambda j: Z[j])
            R = q["rollouts"][i]
            c = np.clip((R["kl"] - q05) / max(q95 - q05, 1e-8), 0, 1)
            ax = axes[r, col]
            ax.plot(sm(c), label="c_t (calibrated conf.)", lw=1); ax.plot(sm(R["v"]), label="v_t (visual engagement)", lw=1)
            ax.set_title(f"q={q['qid']} rollout {R['rollout_id']} ({'correct' if want else 'incorrect'}) Z={Z[i]:.3f} ans={q['reps'][q['group'][i]]}", fontsize=8)
            ax.set_xlabel("generated token (15-token moving avg)")
            if r == 0 and col == 0:
                ax.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(FIG_DIR / "illustrative_trajectories_mathvista.png", dpi=200); plt.close(fig)

In [12]:
if RUN_SIGNALS:
    SIG = {name: signal_analysis(name, DATA[name]) for name in DATASETS}


[MATH] signal analysis (held-out, N=32):
              score  top_minus_bottom_all  spearman_score_correct  spearman_score_length  spearman_length_correct  top_minus_bottom_mixed_only  n_mixed_questions
         mean_logp                0.0783                  0.1341                 0.2245                  -0.0763                       0.1335                469
    self_certainty                0.1197                  0.1979                -0.1611                  -0.0763                       0.2042                469
shortest (−length)                0.0456                  0.0763                -1.0000                  -0.0763                       0.0778                469

[MathVista] signal analysis (held-out, N=32):
              score  top_minus_bottom_all  spearman_score_correct  spearman_score_length  spearman_length_correct  top_minus_bottom_mixed_only  n_mixed_questions
   calibrated_conf                0.0386                  0.0489                 0.3469                 

## 6 · Adaptive sampling (agreement-based and quality-aware stopping), thresholds chosen on dev only

In [13]:
def simulate_policy(Q, feats, tau, gamma=None):
    """Returns per (qid, order) outcome. tau=None → always 32 (fallback)."""
    out = []
    for q in Q:
        for o in range(N_ORDERS):
            stop = 32
            if tau is not None:
                for n in CHECK_AT:
                    f = feats[(q["qid"], o, n)]
                    ok = f["unique_lead"] and f["sc"] != 0 and f["sc_votes"] / n >= tau - 1e-12
                    if ok and gamma is not None:
                        ok = (f["q_winner"] == f["sc"]) and (f["q_margin"] >= gamma - 1e-12)
                    if ok:
                        stop = n; break
            f = feats[(q["qid"], o, stop)]
            f32 = feats[(q["qid"], o, 32)]
            out.append(dict(qid=q["qid"], split=q["split"], order=o, stop=stop, cost=f["cost"], answer=q["reps"][f["sc"]],
                            correct=bool(q["group_correct"][f["sc"]]), correct_fixed32=bool(q["group_correct"][f32["sc"]]),
                            pool_correct_frac=float(q["correct"].mean())))
    return pd.DataFrame(out)

def agg(d):
    per_q = d.groupby("qid").agg(acc=("correct", "mean"), cost=("cost", "mean"), n=("stop", "mean"))
    return dict(accuracy=per_q["acc"].mean(), mean_cost=per_q["cost"].mean(), mean_rollouts=per_q["n"].mean())

def run_adaptive():
    results = {}
    for name in DATASETS:
        Q = DATA[name]; feats = OFF[name][1]
        Qdev = [q for q in Q if q["split"] == "dev"]; Qev = [q for q in Q if q["split"] == "eval"]
        fixed32_dev = agg(simulate_policy(Qdev, feats, None))["accuracy"]
        grid = []
        for family, settings in [("agreement", [(t, None) for t in TAUS]),
                                 ("quality", [(t, g) for t in TAUS for g in GAMMAS])]:
            for t, g in settings + [(None, None)]:
                a = agg(simulate_policy(Qdev, feats, t, g))
                grid.append(dict(family=family, tau=t, gamma=g, **a, feasible=a["accuracy"] >= fixed32_dev - 0.01 - 1e-12))
        grid = pd.DataFrame(grid); grid.to_csv(TAB_DIR / f"adaptive_dev_grid_{name}.csv", index=False)
        chosen = {}
        for family in ("agreement", "quality"):
            g = grid[(grid.family == family) & grid.feasible].copy()
            g["tau_s"] = g["tau"].fillna(9.0); g["gamma_s"] = g["gamma"].fillna(9.0)
            best = g.sort_values(["mean_cost", "accuracy", "tau_s", "gamma_s"], ascending=[True, False, True, True]).iloc[0]
            chosen[family] = (None if pd.isna(best["tau"]) else float(best["tau"]),
                              None if pd.isna(best["gamma"]) else float(best["gamma"]))
        jdump(dict(fixed32_dev_accuracy=fixed32_dev, chosen=chosen,
                   score="self-certainty" if name == "MATH" else "VisConf"), OUT / f"adaptive_frozen_{name}.json")
        # ---------------- held-out (frozen thresholds) ----------------
        rows, per = [], {}
        for family, (t, g) in chosen.items():
            d = simulate_policy(Qev, feats, t, g if family == "quality" else None)
            per[family] = d
            a = agg(d)
            dist = d["stop"].value_counts(normalize=True).reindex([4, 8, 16, 32]).fillna(0).to_dict()
            early_wrong = d[(d.stop < 32) & ~d.correct]
            rows.append(dict(policy=f"{family} (τ={t}, γ={g})", **a, **{f"stop@{k}": v for k, v in dist.items()},
                             incorrect_early_stop_rate=float(len(early_wrong) / len(d)),
                             early_wrong_recoverable_by_fixed32=float(early_wrong["correct_fixed32"].mean()) if len(early_wrong) else 0.0))
            d.to_csv(TAB_DIR / f"adaptive_eval_{name}_{family}.csv", index=False)
        df = OFF[name][0]
        for N in BUDGETS:
            s = df[(df.split == "eval") & (df.method == "SC") & (df.N == N)].groupby("qid").agg(acc=("correct", "mean"), cost=("cost", "mean"))
            rows.append(dict(policy=f"fixed SC N={N}", accuracy=s["acc"].mean(), mean_cost=s["cost"].mean(), mean_rollouts=N))
        res = pd.DataFrame(rows); res.to_csv(TAB_DIR / f"adaptive_eval_{name}.csv", index=False)
        # retrospective: stopping vs. pool correctness (analysis only)
        for family, d in per.items():
            d["pool_bin"] = pd.cut(d["pool_correct_frac"], [-0.01, 0.0, 0.25, 0.5, 0.75, 0.99, 1.0])
            d.groupby("pool_bin", observed=True).agg(mean_stop=("stop", "mean"), accuracy=("correct", "mean"),
                                                     n=("qid", "size")).to_csv(TAB_DIR / f"adaptive_vs_pool_{name}_{family}.csv")
        print(f"\n[{name}] adaptive (dev-chosen {chosen}; fixed-32 dev acc {fixed32_dev:.4f}):\n", res.round(4).to_string(index=False))
        results[name] = (res, per, chosen)
    return results

In [14]:
if RUN_ADAPTIVE:
    ADA = run_adaptive()


[MATH] adaptive (dev-chosen {'agreement': (0.625, None), 'quality': (0.5, 0.0)}; fixed-32 dev acc 0.8250):
                      policy  accuracy  mean_cost  mean_rollouts  stop@4  stop@8  stop@16  stop@32  incorrect_early_stop_rate  early_wrong_recoverable_by_fixed32
agreement (τ=0.625, γ=None)    0.7364 10138.7711        11.8230  0.6630  0.0584   0.0132   0.2654                     0.0825                              0.0773
     quality (τ=0.5, γ=0.0)    0.7318  7663.0554         9.0075  0.7636  0.0581   0.0135   0.1648                     0.1440                              0.1137
               fixed SC N=1    0.6435   669.7726         1.0000     NaN     NaN      NaN      NaN                        NaN                                 NaN
               fixed SC N=2    0.6435  1339.8655         2.0000     NaN     NaN      NaN      NaN                        NaN                                 NaN
               fixed SC N=4    0.6920  2679.6941         4.0000     NaN     NaN      N

## 7 · Live generation on 50 MATH held-out questions: single-long vs 3-parallel (SC) vs sequential refinement
Qwen2.5-3B-Instruct FP16, batch 1, T=0.7 / top-p 0.8 / top-k 20, native chat template, the fixed system message and instruction. Each call is independently seeded. The first parallel response is reused as the refinement's initial answer. Resumable per question.

In [15]:
def live_generation():
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    Q = {q["qid"]: q for q in DATA["MATH"]}
    live_ids = jload(OUT / "splits_MATH.json")["live"]
    assert all(Q[i]["problem"] for i in live_ids), "problem text not found in the supplied files — set COLUMN_OVERRIDES['MATH']['problem']"
    print("first live problem text (check it is the raw problem, not a full chat prompt):\n", Q[live_ids[0]]["problem"][:400])
    path = OUT / "live_records.jsonl"
    done = {r["qid"] for r in read_jsonl(path)}
    if len(done) == len(live_ids):
        return pd.DataFrame(read_jsonl(path))
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(LIVE_MODEL, revision=LIVE_REVISION)
    model = AutoModelForCausalLM.from_pretrained(LIVE_MODEL, revision=LIVE_REVISION, dtype=torch.float16,
                                                 attn_implementation="sdpa").to(dev).eval()
    eos = model.generation_config.eos_token_id
    eos = set(eos if isinstance(eos, list) else [eos])

    def gen(messages, cap, seed):
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        ids = tok(prompt, return_tensors="pt", add_special_tokens=False).input_ids.to(dev)
        torch.manual_seed(seed)
        if dev == "cuda":
            torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
        t = time.time()
        with torch.no_grad():
            out = model.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=cap, pad_token_id=tok.pad_token_id or tok.eos_token_id, **GEN_KW)
        if dev == "cuda":
            torch.cuda.synchronize()
        el = time.time() - t
        new = out[0, ids.shape[1]:].tolist()
        n_out = next((k + 1 for k, x in enumerate(new) if x in eos), len(new))
        ended = any(x in eos for x in new[:n_out])
        text = tok.decode(new[:n_out], skip_special_tokens=True)
        ans = last_boxed(text)
        return dict(prompt=prompt, response=text, seed=seed, input_tokens=int(ids.shape[1]), output_tokens=int(n_out),
                    truncated=not ended, time_s=el, extracted=ans,
                    peak_gpu_gb=torch.cuda.max_memory_allocated() / 1024**3 if dev == "cuda" else 0.0)

    for k, qid in enumerate(live_ids):
        if qid in done:
            continue
        q = Q[qid]
        user = q["problem"] + LIVE_JOIN + LIVE_SUFFIX
        base = [{"role": "system", "content": LIVE_SYSTEM}, {"role": "user", "content": user}]
        s0 = SEED * 1000 + k * 10
        single = gen(base, CAP_SINGLE, s0 + 1)
        par = [gen(base, CAP_PARALLEL, s0 + 2 + j) for j in range(3)]
        revise_msgs = base + [{"role": "assistant", "content": par[0]["response"]}, {"role": "user", "content": REVIEW_PROMPT}]
        rev = gen(revise_msgs, CAP_REVISION, s0 + 6)
        append_jsonl(path, dict(qid=qid, k=k, gold=q["gold"], single=single, parallel=par, revision=rev))
        print(f"  live {k+1}/{len(live_ids)} q={qid} | tokens single={single['output_tokens']} "
              f"par={[p['output_tokens'] for p in par]} rev={rev['output_tokens']}")
    del model; gc.collect()
    return pd.DataFrame(read_jsonl(path))

def score_live(recs):
    rows, trans, cases = [], Counter(), []
    for r in recs.to_dict("records"):
        gp = r["gold"]
        def correct(a):
            if is_missing(a):
                return False
            g, reps, parsed = group_math([a]); return bool(grade_math(reps, parsed, gp)[g[0]])
        s = r["single"]; par = r["parallel"]; rev = r["revision"]
        gid, reps, parsed = group_math([p["extracted"] for p in par])
        g_sc, _ = sc_select(gid)
        par_ans = reps[g_sc] if g_sc != 0 else None
        init_ok, rev_ok = correct(par[0]["extracted"]), correct(rev["extracted"])
        trans[("C" if init_ok else "W") + "→" + ("C" if rev_ok else "W")] += 1
        rows.append(dict(qid=r["qid"], strategy="single_long", correct=correct(s["extracted"]), extracted=not is_missing(s["extracted"]),
                         truncated_any=s["truncated"], input_tokens=s["input_tokens"], output_tokens=s["output_tokens"],
                         runtime_s=s["time_s"], peak_gpu_gb=s["peak_gpu_gb"]))
        rows.append(dict(qid=r["qid"], strategy="parallel_3_SC", correct=correct(par_ans) if par_ans else False,
                         extracted=par_ans is not None, truncated_any=any(p["truncated"] for p in par),
                         truncated_frac=np.mean([p["truncated"] for p in par]),
                         input_tokens=sum(p["input_tokens"] for p in par), output_tokens=sum(p["output_tokens"] for p in par),
                         runtime_s=sum(p["time_s"] for p in par), peak_gpu_gb=max(p["peak_gpu_gb"] for p in par)))
        rows.append(dict(qid=r["qid"], strategy="sequential_refinement", correct=rev_ok, extracted=not is_missing(rev["extracted"]),
                         truncated_any=par[0]["truncated"] or rev["truncated"], initial_correct=init_ok,
                         input_tokens=par[0]["input_tokens"] + rev["input_tokens"], output_tokens=par[0]["output_tokens"] + rev["output_tokens"],
                         runtime_s=par[0]["time_s"] + rev["time_s"], peak_gpu_gb=max(par[0]["peak_gpu_gb"], rev["peak_gpu_gb"])))
        rows.append(dict(qid=r["qid"], strategy="initial_only (1×1024)", correct=init_ok, extracted=not is_missing(par[0]["extracted"]),
                         truncated_any=par[0]["truncated"], input_tokens=par[0]["input_tokens"], output_tokens=par[0]["output_tokens"],
                         runtime_s=par[0]["time_s"], peak_gpu_gb=par[0]["peak_gpu_gb"]))
        if init_ok != rev_ok:
            cases.append(dict(qid=r["qid"], transition=("C" if init_ok else "W") + "→" + ("C" if rev_ok else "W"),
                              gold=gp, initial=par[0]["extracted"], revised=rev["extracted"], revision_text=rev["response"][:1500]))
    df = pd.DataFrame(rows)
    summ = df.groupby("strategy").agg(accuracy=("correct", "mean"), extraction_rate=("extracted", "mean"),
                                      truncation_rate=("truncated_any", "mean"), mean_input_tokens=("input_tokens", "mean"),
                                      mean_output_tokens=("output_tokens", "mean"), mean_runtime_s=("runtime_s", "mean"),
                                      peak_gpu_gb=("peak_gpu_gb", "max")).reset_index()
    summ.to_csv(TAB_DIR / "live_strategies.csv", index=False); df.to_csv(TAB_DIR / "live_per_question.csv", index=False)
    jdump(dict(transitions=dict(trans), n=len(recs)), TAB_DIR / "live_revision_transitions.json")
    jdump(cases, TAB_DIR / "live_revision_cases.json")
    piv = df.pivot(index="qid", columns="strategy", values="correct").astype(float)
    boots = {f"{a} − {b}": paired_boot(piv[a], piv[b]) for a, b in [("parallel_3_SC", "single_long"),
             ("sequential_refinement", "single_long"), ("sequential_refinement", "parallel_3_SC")]}
    jdump(boots, TAB_DIR / "live_bootstrap.json")
    print(summ.round(3).to_string(index=False)); print("revision transitions:", dict(trans)); print(boots)
    return summ

In [16]:
if RUN_LIVE:
    LIVE = live_generation()
    LIVE_SUMMARY = score_live(LIVE)

first live problem text (check it is the raw problem, not a full chat prompt):
 The arithmetic mean (or average) of $A$, $B$ and $C$ is 10. The value of $A$ is six less than the value of $B$, and the value of $C$ is three more than the value of $B$. What is the value of $C$?
             strategy  accuracy  extraction_rate  truncation_rate  mean_input_tokens  mean_output_tokens  mean_runtime_s  peak_gpu_gb
initial_only (1×1024)      0.56             0.84             0.16             108.38              606.06          30.717        5.935
        parallel_3_SC      0.60             0.84             0.24             325.14             1861.30          94.214        5.935
sequential_refinement      0.56             0.96             0.18             859.98             1221.04          63.993        6.174
          single_long      0.60             0.98             0.02             108.38              715.86          36.695        6.084
revision transitions: {'C→C': 27, 'W→W': 21, 'C→W': 1,

## 8 · Compute allocation figures and failure cases

In [17]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.ticker
import matplotlib.pyplot as plt

def figures():
    for name in DATASETS:
        df = OFF[name][0]
        _, s = summarize(df, "eval")
        fig, ax = plt.subplots(1, 3, figsize=(16, 4))
        cols = plt.cm.tab20(np.linspace(0, 1, 20))
        for mi, m in enumerate(method_names(name)):
            d = s[s.method == m].sort_values("N")
            ls = "--" if m in ("FirstAns", "Shortest") else (":" if m.endswith("+RWC") else "-")
            ax[0].plot(d["N"], d["accuracy"], ls, marker="o", ms=3, label=m, color=cols[(2 * mi) % 20 + (mi >= 10)])
            ax[1].plot(d["mean_cost"], d["accuracy"], ls, marker="o", ms=3, label=m, color=cols[(2 * mi) % 20 + (mi >= 10)])
        d = s[s.method == "SC"].sort_values("N")
        ax[0].plot(d["N"], d["oracle"], "k:", marker="s", label="oracle coverage")
        ax[2].plot(d["N"], d["distinct_groups"], "o-")
        ax[2].set_title("distinct answer groups"); ax[2].set_xscale("log", base=2); ax[2].set_xlabel("N candidates")
        if RUN_ADAPTIVE and name in globals().get("ADA", {}):
            res = ADA[name][0]
            for _, r in res[~res.policy.str.startswith("fixed")].iterrows():
                ax[1].scatter(r["mean_cost"], r["accuracy"], marker="*", s=150, zorder=5, label=f"adaptive {r['policy'].split(' ')[0]}")
        ax[0].set_xscale("log", base=2); ax[0].set_xlabel("N candidates"); ax[0].set_ylabel("held-out accuracy")
        ax[1].set_xscale("log"); ax[1].xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
        ax[1].set_xlabel("mean generated tokens / question"); ax[0].legend(fontsize=7, ncol=2)
        ax[0].set_title(f"{name}: accuracy vs N"); ax[1].set_title("accuracy vs token cost")
        ax[1].legend(fontsize=6, ncol=2)
        fig.tight_layout(); fig.savefig(FIG_DIR / f"compute_allocation_{name}.png", dpi=200); plt.close(fig)
        fd = failure_decomposition(df)
        fig, ax = plt.subplots(figsize=(6, 3.5))
        d = fd[fd.method == "SC"].sort_values("N")
        ax.stackplot(np.log2(d["N"]), d["correct"], d["correct_available_but_missed"], d["no_correct_candidate"],
                     labels=["correct", "correct available, missed (selection)", "no correct candidate (sampling)"], alpha=0.8)
        ax.set_xticks(np.log2(BUDGETS)); ax.set_xticklabels(BUDGETS); ax.set_xlabel("N"); ax.set_title(f"{name}: SC outcome decomposition")
        ax.legend(fontsize=7, loc="lower left"); fig.tight_layout(); fig.savefig(FIG_DIR / f"failure_decomposition_{name}.png", dpi=200); plt.close(fig)

def failure_cases(n=3):
    """Selection rule: first n held-out questions (split-permutation order) where, at N=32 order 0, SC and
    SelfCert+RWC (MATH) / VisConf+RWC (MathVista) choose different answers; and first n questions where the
    agreement policy stopped early with a wrong answer that fixed-32 SC would have fixed."""
    lines = []
    for name in DATASETS:
        df = OFF[name][0]; Q = {q["qid"]: q for q in DATA[name]}
        order = jload(OUT / f"splits_{name}.json")["eval"]
        alt = "SelfCert+RWC" if name == "MATH" else "VisConf+RWC"
        d = df[(df.N == 32) & (df.order == 0) & df.method.isin(["SC", alt])].pivot(index="qid", columns="method", values=["answer", "correct"])
        dis = [q for q in order if q in d.index and d.loc[q, ("answer", "SC")] != d.loc[q, ("answer", alt)]][:n]
        lines.append(f"## {name}: selection disagreements (SC vs {alt}, N=32, order 0)\n")
        for q in dis:
            votes = Counter(Q[q]["reps"][g] for g in Q[q]["group"])
            lines.append(f"- q={q} gold={Q[q]['gold']} | SC → {d.loc[q, ('answer', 'SC')]} ({'✓' if d.loc[q, ('correct', 'SC')] else '✗'}) | "
                         f"{alt} → {d.loc[q, ('answer', alt)]} ({'✓' if d.loc[q, ('correct', alt)] else '✗'}) | votes {dict(votes.most_common(5))}")
        if RUN_ADAPTIVE and name in globals().get("ADA", {}):
            ag = ADA[name][1]["agreement"]
            bad = ag[(ag.stop < 32) & ~ag.correct & ag.correct_fixed32]
            lines.append(f"\n## {name}: premature stops (agreement policy) later fixed by fixed-32 SC\n")
            for q in [x for x in order if x in set(bad.qid)][:n]:
                b = bad[bad.qid == q].iloc[0]
                lines.append(f"- q={q} order={b['order']} stopped at n={b['stop']} with '{b['answer']}' (gold {Q[q].get('gold_key', Q[q]['gold'])}); "
                             f"pool correct fraction {b['pool_correct_frac']:.2f}")
        lines.append("")
    if (TAB_DIR / "live_revision_cases.json").exists():
        lines.append("## Live refinement transitions (all W→C and C→W cases are listed in tables/live_revision_cases.json)\n")
        for c in jload(TAB_DIR / "live_revision_cases.json")[:n]:
            lines.append(f"- q={c['qid']} {c['transition']}: initial={c['initial']} → revised={c['revised']} (gold {c['gold']})")
    (TAB_DIR / "failure_cases.md").write_text("\n".join(lines))
    print("\n".join(lines[:40]))

In [18]:
if RUN_ANALYSIS:
    figures()
    failure_cases()
    print("tables →", TAB_DIR, "| figures →", FIG_DIR)

## MATH: selection disagreements (SC vs SelfCert+RWC, N=32, order 0)

- q=test:2460 gold=154 | SC → 58 (✗) | SelfCert+RWC → 154 (✓) | votes {'154': 11, '58': 11, 'NO_ANSWER': 3, '45': 1, '98': 1}
- q=test:2892 gold=-71 | SC → 1 (✗) | SelfCert+RWC → 0 (✗) | votes {'0': 4, '1': 4, '23': 2, '-1': 1, 'NO_ANSWER': 1}
- q=test:2948 gold=1+\sqrt 2,\ 1-\sqrt 2,\ 2+\sqrt 3,\ 2-\sqrt 3 | SC → 1 (✗) | SelfCert+RWC → 2 + \sqrt{3}, 2 - \sqrt{3}, \frac{3 + \sqrt{13 + 4\sqrt{3}}}{2}, \frac{3 - \sqrt{13 + 4\sqrt{3}}}{2}, \frac{3 + \sqrt{13 - 4\sqrt{3}}}{2}, \frac{3 - \sqrt{13 - 4\sqrt{3}}}{2} (✗) | votes {'1': 5, '1 \\text{ and } 2': 4, '2 + \\sqrt{3}, 2 - \\sqrt{3}': 3, '2 + \\sqrt{3}, 2 - \\sqrt{3}, \\frac{3 + \\sqrt{13 + 4\\sqrt{3}}}{2}, \\frac{3 - \\sqrt{13 + 4\\sqrt{3}}}{2}, \\frac{3 + \\sqrt{13 - 4\\sqrt{3}}}{2}, \\frac{3 - \\sqrt{13 - 4\\sqrt{3}}}{2}': 3, 'NO_ANSWER': 2}

## MATH: premature stops (agreement policy) later fixed by fixed-32 SC

- q=test:2460 order=7 stopped at n=4 with '58' (gold